In [0]:
%python
bronze_path   = 'bikestore.bronze'
silver_path   = 'bikestore.silver'
gold_path     = 'bikestore.gold'
resource_path = "/Volumes/bikestore/resource/origem"
resource_path_volume = '/Volumes/bikestore/resource/origem'

In [0]:
%python
# criar varias tabelas temporárias de forma prática

bronze_map = {
    # View_name --------------- path
    # "tmp_bronze_brands": f"{bronze_path}.brands",
    # "tmp_bronze_categories": f"{bronze_path}.categories",
    # "tmp_bronze_customers": f"{bronze_path}.customers",
    "tmp_bronze_order_items": f"{bronze_path}.order_items",
    "tmp_bronze_orders": f"{bronze_path}.orders",
    #"tmp_bronze_products": f"{bronze_path}.products",
    "tmp_bronze_staffs": f"{bronze_path}.staffs",
    #"tmp_bronze_stocks": f"{bronze_path}.stocks",
    "tmp_bronze_stores": f"{bronze_path}.stores",
}

for view_name, path in bronze_map.items():
    (spark.table(path).createOrReplaceTempView(view_name))

In [0]:
%python
from pyspark.sql.functions import current_timestamp, lit, year, month

current_ts = current_timestamp()

df_order_silver = spark.sql("""
    with orders_items as (
    select
        oi.order_id,
        oi.item_id,
        oi.product_id,
        oi.quantity,
        oi.list_price,
        oi.discount,
        round(oi.quantity * (oi.list_price * (1 - oi.discount)), 2) total_sale,
        oi.bronze_source_file as source_order_items_file
    from
        tmp_bronze_order_items oi
    )
    
    select
    oi.item_id,
    o.order_id,
    o.customer_id,
    CASE
        WHEN o.order_status = 1 THEN 'Pending'
        WHEN o.order_status = 2 THEN 'Processing'
        WHEN o.order_status = 3 THEN 'Shipped'
        WHEN o.order_status = 4 THEN 'Delivered'
        ELSE 'Unknown'
    END as order_status,
    o.order_date,
    o.required_date,
    o.shipped_date,
    s.store_name,
    s.city,
    s.state,
    f.first_name,
    f.email,
    f.active,
    oi.product_id,
    oi.quantity,
    oi.list_price,
    oi.discount,
    oi.total_sale,
    --Metadata
    o.bronze_source_file as source_orders_file,
    s.bronze_source_file as source_store_file,
    f.bronze_source_file as source_staffs_file,
    oi.source_order_items_file
    --o.order_status,
    -- o.staff_id,
    -- f.staff_id,
    from
    tmp_bronze_orders as o
        LEFT JOIN tmp_bronze_stores as s
        ON o.store_id = s.store_id
        LEFT JOIN tmp_bronze_staffs as f
        ON o.staff_id = f.staff_id
        LEFT JOIN orders_items as oi
        ON o.order_id = oi.order_id
""")\
  .withColumn("year",year("order_date").cast("int"))\
  .withColumn("month", month("order_date").cast("int"))\
  .withColumn("ingestion_timestamp", current_ts) \
  .withColumn("created_at", current_ts) \
  .withColumn("updated_at", lit(None).cast("timestamp"))

In [0]:
%python
#salvar em parquet como delta na camada silver
df_order_silver.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .partitionBy("year","month")\
    .saveAsTable(f"{silver_path}.orders")